# **Deep Learning Assignment 1**

# Task 2.1a — Derivation

The loss function is defined as

$$
L = \frac{1}{N}\sum_{i=1}^{N} (\hat{y}_i - y_i)^2
$$

where:
- $\hat{y}_i$ is the predicted value  
- $y_i$ is the actual value  
- $N$ is the number of samples  

---
### Step 1: Differentiate one term

Consider a single term in the summation:

$$
(\hat{y}_i - y_i)^2
$$

Since $y_i$ is constant with respect to $\hat{y}_i$, we differentiate:

$$
\frac{\partial}{\partial \hat{y}_i}(\hat{y}_i - y_i)^2 = 2(\hat{y}_i - y_i)
$$

---
### Step 2: Differentiate the full summation

Since we differentiate with respect to $\hat{y}_i$, only the $i$-th term contributes and all other terms are constant. Therefore,

$$
\frac{\partial}{\partial \hat{y}_i} \sum_{j=1}^{N} (\hat{y}_j - y_j)^2
= 2(\hat{y}_i - y_i)
$$

Including the averaging factor $\frac{1}{N}$:

$$
\frac{\partial L}{\partial \hat{y}_i} = \frac{2}{N}(\hat{y}_i - y_i)
$$

---
### Step 3: Write in vector form

Applying this to all components, we obtain:

$$
\frac{\partial L}{\partial \hat{y}} =
\frac{2}{N}
\begin{bmatrix}
\hat{y}_1 - y_1 \\
\hat{y}_2 - y_2 \\
\vdots \\
\hat{y}_N - y_N
\end{bmatrix}
$$

Equivalently, in compact vector notation:

$$
\frac{\partial L}{\partial \hat{y}} = \frac{2}{N}(\hat{y} - y)
$$


### Interpretation

- If $\hat{y}_i > y_i$, the gradient is positive → prediction is too high  
- If $\hat{y}_i < y_i$, the gradient is negative → prediction is too low  

This gradient is used during backpropagation to update model parameters.

In [97]:
import numpy as np

def compute_dL_dyhat(y_hat, y):
    N = y_hat.shape[0]
    return (2.0 / N) * (y_hat - y)


In [98]:

np.random.seed(0)

N = 7
y_hat = np.random.randn(N)
y = np.random.randn(N)

# student function
dL_dyhat = compute_dL_dyhat(y_hat, y)

# shape checks
assert isinstance(dL_dyhat, np.ndarray), "dL_dyhat must be a NumPy array"
assert dL_dyhat.shape == y_hat.shape, f"Shape mismatch: expected {y_hat.shape}, got {dL_dyhat.shape}"

# value check (closed-form)
expected = (2.0 / N) * (y_hat - y)
assert np.allclose(dL_dyhat, expected, rtol=1e-10, atol=1e-12), "Incorrect dL/dy_hat"

print("+++ Task 2.1b checks passed.")

+++ Task 2.1b checks passed.


## Task 2.2a — Derivation

We consider the affine map

$$
\hat{y} = Hv + c
$$

where:
- $H \in \mathbb{R}^{N \times h}$
- $v \in \mathbb{R}^{h}$
- $c \in \mathbb{R}$
- $\hat{y} \in \mathbb{R}^{N}$

We are given the upstream gradient

$$
\frac{\partial L}{\partial \hat{y}} \in \mathbb{R}^{N}
$$

and want to derive expressions for

$$
\frac{\partial L}{\partial H},
\frac{\partial L}{\partial v},
\frac{\partial L}{\partial c}.
$$

---

### Step 1: First-order variation

Since the map is affine,

$$
\delta \hat{y} = \delta H \, v + H \, \delta v + \delta c.
$$

---

### Step 2: Propagate into the loss

By definition,

$$
\delta L = \left\langle \frac{\partial L}{\partial \hat{y}}, \delta \hat{y} \right\rangle.
$$

Substituting $\delta \hat{y}$ gives

$$
\delta L
=
\left\langle \frac{\partial L}{\partial \hat{y}}, \delta H\,v \right\rangle
+
\left\langle \frac{\partial L}{\partial \hat{y}}, H\,\delta v \right\rangle
+
\left\langle \frac{\partial L}{\partial \hat{y}}, \delta c \right\rangle.
$$

---

### Step 3: Isolate $\delta H$

For the first term, we rewrite it so that $\delta H$ appears alone. This gives

$$
\left\langle \frac{\partial L}{\partial \hat{y}}, \delta H\,v \right\rangle
=
\left\langle \frac{\partial L}{\partial \hat{y}}\,v^\top, \delta H \right\rangle.
$$

Therefore, the coefficient of $\delta H$ is

$$
\boxed{
\frac{\partial L}{\partial H}
=
\frac{\partial L}{\partial \hat{y}}\,v^\top
}
$$

Shape check:

$$
\frac{\partial L}{\partial \hat{y}} \in \mathbb{R}^{N}, \qquad
v^\top \in \mathbb{R}^{1 \times h}
$$

so

$$
\frac{\partial L}{\partial H} \in \mathbb{R}^{N \times h},
$$

which matches the shape of $H$.

---

### Step 4: Isolate $\delta v$

For the second term, use the identity

$$
\langle a, Bx \rangle = \langle B^\top a, x \rangle
$$

with $a = \frac{\partial L}{\partial \hat{y}}$, $B = H$, and $x = \delta v$:

$$
\left\langle \frac{\partial L}{\partial \hat{y}}, H\,\delta v \right\rangle
=
\left\langle H^\top \frac{\partial L}{\partial \hat{y}}, \delta v \right\rangle.
$$

Therefore, the coefficient of $\delta v$ is

$$
\boxed{
\frac{\partial L}{\partial v}
=
H^\top \frac{\partial L}{\partial \hat{y}}
}
$$

Shape check:

$$
H^\top \in \mathbb{R}^{h \times N}, \qquad
\frac{\partial L}{\partial \hat{y}} \in \mathbb{R}^{N}
$$

so

$$
\frac{\partial L}{\partial v} \in \mathbb{R}^{h},
$$

which matches the shape of $v$.

---
### Step 5: Isolate $\delta c$

Since $c$ is a scalar added to every component of $\hat{y}$, the third term becomes

$$
\left\langle \frac{\partial L}{\partial \hat{y}}, \delta c \right\rangle
=
\sum_{i=1}^N \frac{\partial L}{\partial \hat{y}_i}\,\delta c
=
\left(\sum_{i=1}^N \frac{\partial L}{\partial \hat{y}_i}\right)\delta c.
$$

Therefore, the coefficient of $\delta c$ is

$$
\boxed{
\frac{\partial L}{\partial c}
=
\sum_{i=1}^N \frac{\partial L}{\partial \hat{y}_i}
}
$$

---

### Final Result

Thus, the gradients are

$$
\boxed{
\frac{\partial L}{\partial H}
=
\frac{\partial L}{\partial \hat{y}}\,v^\top
}
$$

$$
\boxed{
\frac{\partial L}{\partial v}
=
H^\top \frac{\partial L}{\partial \hat{y}}
}
$$

$$
\boxed{
\frac{\partial L}{\partial c}
=
\sum_{i=1}^N \frac{\partial L}{\partial \hat{y}_i}
}
$$

### Interpretation

This result can be understood as follows:

- $\frac{\partial L}{\partial H}$ sends the error backward to the previous layer
- $\frac{\partial L}{\partial v}$ measures how much each weight contributed to the error
- $\frac{\partial L}{\partial c}$ adds all output error contributions because the same scalar bias $c$ affects every prediction



## Task 2.2b — Implementation

Using the gradients derived in Task 2.2a, we implement the backward pass for the output affine map
$$
\hat{y} = Hv + c
$$

The required gradients are:
$$
\frac{\partial L}{\partial H} = \frac{\partial L}{\partial \hat{y}} v^\top
$$

$$
\frac{\partial L}{\partial v} = H^\top \frac{\partial L}{\partial \hat{y}}
$$

$$
\frac{\partial L}{\partial c} = \sum_{i=1}^N \frac{\partial L}{\partial \hat{y}_i}
$$

In [99]:
def backward_output_affine(dL_dyhat, H, v):
    dL_dH = np.outer(dL_dyhat, v)
    dL_dv = H.T @ dL_dyhat
    dL_dc = np.sum(dL_dyhat)
    return dL_dH, dL_dv, dL_dc

In [100]:
def loss_from_yhat(y_hat, y):
    return np.mean((y_hat - y) ** 2)

def finite_diff_grad_H_v_c(H, v, c, y, eps=1e-6):
    N, h = H.shape

    # grads
    gH = np.zeros_like(H)
    gv = np.zeros_like(v)
    gc = 0.0

    # H
    for i in range(N):
        for j in range(h):
            H_p = H.copy()
            H_m = H.copy()
            H_p[i, j] += eps
            H_m[i, j] -= eps
            L_p = loss_from_yhat(H_p @ v + c, y)
            L_m = loss_from_yhat(H_m @ v + c, y)
            gH[i, j] = (L_p - L_m) / (2 * eps)

    # v
    for j in range(h):
        v_p = v.copy()
        v_m = v.copy()
        v_p[j] += eps
        v_m[j] -= eps
        L_p = loss_from_yhat(H @ v_p + c, y)
        L_m = loss_from_yhat(H @ v_m + c, y)
        gv[j] = (L_p - L_m) / (2 * eps)

    # c
    L_p = loss_from_yhat(H @ v + (c + eps), y)
    L_m = loss_from_yhat(H @ v + (c - eps), y)
    gc = (L_p - L_m) / (2 * eps)

    return gH, gv, gc

np.random.seed(1)
N, h = 5, 4
H = np.random.randn(N, h)
v = np.random.randn(h)
c = np.random.randn()
y = np.random.randn(N)

# upstream from Task 2.1
y_hat = H @ v + c
dL_dyhat = (2.0 / N) * (y_hat - y)

# student function
dL_dH, dL_dv, dL_dc = backward_output_affine(dL_dyhat, H, v)

# shape checks
assert dL_dH.shape == H.shape, f"dL_dH shape mismatch: expected {H.shape}, got {dL_dH.shape}"
assert dL_dv.shape == v.shape, f"dL_dv shape mismatch: expected {v.shape}, got {dL_dv.shape}"
assert np.isscalar(dL_dc) or (isinstance(dL_dc, np.ndarray) and dL_dc.shape == ()), "dL_dc must be a scalar"

# value checks vs finite differences
gH_num, gv_num, gc_num = finite_diff_grad_H_v_c(H, v, c, y, eps=1e-6)

def rel_err(a, b):
    return np.linalg.norm(a - b) / (np.linalg.norm(a) + np.linalg.norm(b) + 1e-12)

errH = rel_err(dL_dH, gH_num)
errv = rel_err(dL_dv, gv_num)
errc = abs(dL_dc - gc_num) / (abs(dL_dc) + abs(gc_num) + 1e-12)

assert errH < 2e-5, f"dL_dH failed finite-diff check (rel err {errH:.2e})"
assert errv < 2e-5, f"dL_dv failed finite-diff check (rel err {errv:.2e})"
assert errc < 2e-5, f"dL_dc failed finite-diff check (rel err {errc:.2e})"

print("+++ Task 2.2b checks passed.")

+++ Task 2.2b checks passed.


## Task 2.3a — Derivation

We consider the activation map

$$
H = \phi(Z), \qquad \phi = \text{ReLU}
$$

where ReLU is applied elementwise:

$$
H_{ij} =
\begin{cases}
Z_{ij}, & Z_{ij} > 0 \\
0, & Z_{ij} \le 0
\end{cases}
$$

We are given the upstream gradient

$$
\frac{\partial L}{\partial H}
$$

and want to compute

$$
\frac{\partial L}{\partial Z}.
$$

---
### Step 1: Derivative of ReLU

Since ReLU is applied elementwise, its derivative is also elementwise:

$$
\frac{\partial H_{ij}}{\partial Z_{ij}} =
\begin{cases}
1, & Z_{ij} > 0 \\
0, & Z_{ij} \le 0
\end{cases}
$$

This can be written as the binary mask

$$
\mathbf{1}_{Z>0}.
$$

---
### Step 2: Apply the chain rule

Using the chain rule elementwise,

$$
\frac{\partial L}{\partial Z_{ij}}
=
\frac{\partial L}{\partial H_{ij}}
\frac{\partial H_{ij}}{\partial Z_{ij}}.
$$

Substituting the derivative of ReLU gives

$$
\frac{\partial L}{\partial Z_{ij}}
=
\frac{\partial L}{\partial H_{ij}} \cdot \mathbf{1}_{Z_{ij}>0}.
$$

---

### Step 3: Write in matrix form

Therefore, for the full matrix,

$$
\boxed{
\frac{\partial L}{\partial Z}
=
\frac{\partial L}{\partial H} \odot \mathbf{1}_{Z>0}
}
$$

where $\odot$ denotes elementwise multiplication.

### Interpretation

- if $Z_{ij} > 0$, the gradient passes through unchanged
- if $Z_{ij} \le 0$, the gradient becomes zero

So ReLU acts like a gate during backpropagation.

## Task 2.3b — Implementation

Using the result from Task 2.3a, the gradient is given by

$$
\frac{\partial L}{\partial Z}
=
\frac{\partial L}{\partial H} \odot \mathbf{1}_{Z>0}
$$

where $\mathbf{1}_{Z > 0}$ is a binary mask that is 1 when $Z > 0$ and 0 otherwise.

This corresponds to elementwise multiplication of the upstream gradient with the ReLU mask.

In [101]:
def backward_activation_relu(dL_dH, Z):
    dL_dZ = dL_dH * (Z > 0)
    return dL_dZ

In [102]:
def relu(Z):
    return np.maximum(Z, 0.0)

def loss_via_relu(Z, dL_dH):
    # Synthetic scalar objective: <dL_dH, H> where H = relu(Z)
    # This makes upstream gradient exactly dL_dH.
    H = relu(Z)
    return float(np.sum(dL_dH * H))

def finite_diff_grad_Z(Z, dL_dH, eps=1e-6):
    gZ = np.zeros_like(Z)
    it = np.nditer(Z, flags=["multi_index"], op_flags=["readwrite"])
    while not it.finished:
        idx = it.multi_index
        Z_p = Z.copy()
        Z_m = Z.copy()
        Z_p[idx] += eps
        Z_m[idx] -= eps
        L_p = loss_via_relu(Z_p, dL_dH)
        L_m = loss_via_relu(Z_m, dL_dH)
        gZ[idx] = (L_p - L_m) / (2 * eps)
        it.iternext()
    return gZ

np.random.seed(2)
Z = np.random.randn(4, 6)
dL_dH = np.random.randn(4, 6)

# student function
dL_dZ = backward_activation_relu(dL_dH, Z)

# shape
assert dL_dZ.shape == Z.shape, f"Shape mismatch: expected {Z.shape}, got {dL_dZ.shape}"

# value vs finite differences (avoid kink exactly at 0 by nudging)
Z_safe = Z.copy()
Z_safe[np.abs(Z_safe) < 1e-3] += 1e-2
dL_dZ_num = finite_diff_grad_Z(Z_safe, dL_dH, eps=1e-6)

def rel_err(a, b):
    return np.linalg.norm(a - b) / (np.linalg.norm(a) + np.linalg.norm(b) + 1e-12)

err = rel_err(dL_dZ, dL_dZ_num)
assert err < 2e-5, f"Failed finite-diff check (rel err {err:.2e})"

print("+++ Task 2.3b checks passed.")

+++ Task 2.3b checks passed.


## Task 2.4a — Derivation

We consider the affine map

$$
Z = XW + b
$$

where:
- $X \in \mathbb{R}^{N \times d}$
- $W \in \mathbb{R}^{d \times h}$
- $b \in \mathbb{R}^{h}$
- $Z \in \mathbb{R}^{N \times h}$

We are given the upstream gradient

$$
\frac{\partial L}{\partial Z} \in \mathbb{R}^{N \times h}
$$

and we aim to derive expressions for:

$$
\frac{\partial L}{\partial X}, \qquad
\frac{\partial L}{\partial W}, \qquad
\frac{\partial L}{\partial b}.
$$

---

### Step 1: First-order variation

Since the map is affine,

$$
\delta Z = \delta X\,W + X\,\delta W + \delta b.
$$

This describes how small changes in $X$, $W$, and $b$ affect the output $Z$.

---

### Step 2: Propagate into the loss

By definition,

$$
\delta L = \left\langle \frac{\partial L}{\partial Z}, \delta Z \right\rangle.
$$

Substituting the expression for $\delta Z$ gives

$$
\delta L
=
\left\langle \frac{\partial L}{\partial Z}, \delta X\,W \right\rangle
+
\left\langle \frac{\partial L}{\partial Z}, X\,\delta W \right\rangle
+
\left\langle \frac{\partial L}{\partial Z}, \delta b \right\rangle.
$$

We now isolate each perturbation.

---
### Step 3: Derivation of $\frac{\partial L}{\partial X}$

Consider the first term:

$$
\left\langle \frac{\partial L}{\partial Z}, \delta X\,W \right\rangle.
$$

To isolate $\delta X$, we use the identity

$$
\langle a, Bx \rangle = \langle a B^\top, x \rangle.
$$

Applying this gives

$$
\left\langle \frac{\partial L}{\partial Z}, \delta X\,W \right\rangle
=
\left\langle \frac{\partial L}{\partial Z} W^\top, \delta X \right\rangle.
$$

Hence,

$$
\boxed{
\frac{\partial L}{\partial X}
=
\frac{\partial L}{\partial Z} W^\top
}
$$

#### Shape check

- $\frac{\partial L}{\partial Z} \in \mathbb{R}^{N \times h}$
- $W^\top \in \mathbb{R}^{h \times d}$

So,

$$
\frac{\partial L}{\partial X} \in \mathbb{R}^{N \times d}
$$

which matches the shape of $X$.

---

### Step 4: Derivation of $\frac{\partial L}{\partial W}$

Now consider the second term:

$$
\left\langle \frac{\partial L}{\partial Z}, X\,\delta W \right\rangle.
$$

Using the identity

$$
\langle a, Bx \rangle = \langle B^\top a, x \rangle,
$$

we rewrite

$$
\left\langle \frac{\partial L}{\partial Z}, X\,\delta W \right\rangle
=
\left\langle X^\top \frac{\partial L}{\partial Z}, \delta W \right\rangle.
$$

Hence,

$$
\boxed{
\frac{\partial L}{\partial W}
=
X^\top \frac{\partial L}{\partial Z}
}
$$

#### Shape check

- $X^\top \in \mathbb{R}^{d \times N}$
- $\frac{\partial L}{\partial Z} \in \mathbb{R}^{N \times h}$

So,

$$
\frac{\partial L}{\partial W} \in \mathbb{R}^{d \times h}
$$

which matches the shape of $W$.

---

### Step 5: Derivation of $\frac{\partial L}{\partial b}$

Consider the third term:

$$
\left\langle \frac{\partial L}{\partial Z}, \delta b \right\rangle.
$$

Since $b$ is added to every row of $Z$, each component of $b$ affects all samples. Therefore,

$$
\delta L
=
\sum_{i=1}^{N} \sum_{j=1}^{h}
\frac{\partial L}{\partial Z_{ij}} \, \delta b_j.
$$

Rewriting by grouping terms:

$$
\delta L
=
\sum_{j=1}^{h}
\left(
\sum_{i=1}^{N} \frac{\partial L}{\partial Z_{ij}}
\right)\delta b_j.
$$

Thus,

$$
\boxed{
\frac{\partial L}{\partial b}
=
\sum_{i=1}^{N} \frac{\partial L}{\partial Z_{i,:}}
}
$$

#### Shape check

Summing over $i$ gives a vector in $\mathbb{R}^{h}$, matching the shape of $b$.

---

### Final Result

$$
\boxed{
\frac{\partial L}{\partial X}
=
\frac{\partial L}{\partial Z} W^\top
}
$$

$$
\boxed{
\frac{\partial L}{\partial W}
=
X^\top \frac{\partial L}{\partial Z}
}
$$

$$
\boxed{
\frac{\partial L}{\partial b}
=
\sum_{i=1}^{N} \frac{\partial L}{\partial Z_{i,:}}
}
$$


### Interpretation

- $\frac{\partial L}{\partial X}$ passes the error backward to the input through the weights
- $\frac{\partial L}{\partial W}$ measures how much each weight contributed to the loss
- $\frac{\partial L}{\partial b}$ sums contributions across all samples since the same bias is applied to every row

## Task 2.4b — Implementation

Using the gradients derived in Task 2.4a, we implement the backward pass for the affine map

$$
Z = XW + b
$$

The required gradients are

$$
\frac{\partial L}{\partial W} = X^\top \frac{\partial L}{\partial Z}
$$

$$
\frac{\partial L}{\partial b} = \sum_{i=1}^{N} \frac{\partial L}{\partial Z_{i,:}}
$$

$$
\frac{\partial L}{\partial X} = \frac{\partial L}{\partial Z} W^\top
$$

In [103]:
def backward_input_affine(dL_dZ, X, W):
    dL_dW = X.T @ dL_dZ
    dL_db = np.sum(dL_dZ, axis=0)
    dL_dX = dL_dZ @ W.T
    return dL_dW, dL_db, dL_dX

In [104]:
def loss_from_Z(Z, dL_dZ):
    # Synthetic scalar objective: <dL_dZ, Z>
    # This makes upstream gradient exactly dL_dZ.
    return float(np.sum(dL_dZ * Z))

def finite_diff_grad_X_W_b(X, W, b, dL_dZ, eps=1e-6):
    N, d = X.shape
    d_, h = W.shape
    assert d_ == d

    # base
    def Z_of(X_, W_, b_):
        return X_ @ W_ + b_

    gX = np.zeros_like(X)
    gW = np.zeros_like(W)
    gb = np.zeros_like(b)

    # X
    for i in range(N):
        for j in range(d):
            X_p = X.copy()
            X_m = X.copy()
            X_p[i, j] += eps
            X_m[i, j] -= eps
            L_p = loss_from_Z(Z_of(X_p, W, b), dL_dZ)
            L_m = loss_from_Z(Z_of(X_m, W, b), dL_dZ)
            gX[i, j] = (L_p - L_m) / (2 * eps)

    # W
    for i in range(d):
        for j in range(h):
            W_p = W.copy()
            W_m = W.copy()
            W_p[i, j] += eps
            W_m[i, j] -= eps
            L_p = loss_from_Z(Z_of(X, W_p, b), dL_dZ)
            L_m = loss_from_Z(Z_of(X, W_m, b), dL_dZ)
            gW[i, j] = (L_p - L_m) / (2 * eps)

    # b
    for j in range(h):
        b_p = b.copy()
        b_m = b.copy()
        b_p[j] += eps
        b_m[j] -= eps
        L_p = loss_from_Z(Z_of(X, W, b_p), dL_dZ)
        L_m = loss_from_Z(Z_of(X, W, b_m), dL_dZ)
        gb[j] = (L_p - L_m) / (2 * eps)

    return gX, gW, gb

np.random.seed(3)
N, d, h = 4, 5, 3
X = np.random.randn(N, d)
W = np.random.randn(d, h)
b = np.random.randn(h)

# arbitrary upstream gradient
dL_dZ = np.random.randn(N, h)

# student function
dL_dW, dL_db, dL_dX = backward_input_affine(dL_dZ, X, W)

# shape checks
assert dL_dW.shape == W.shape, f"dL_dW shape mismatch: expected {W.shape}, got {dL_dW.shape}"
assert dL_db.shape == b.shape, f"dL_db shape mismatch: expected {b.shape}, got {dL_db.shape}"
assert dL_dX.shape == X.shape, f"dL_dX shape mismatch: expected {X.shape}, got {dL_dX.shape}"

# value checks vs finite differences
gX_num, gW_num, gb_num = finite_diff_grad_X_W_b(X, W, b, dL_dZ, eps=1e-6)

def rel_err(a, b):
    return np.linalg.norm(a - b) / (np.linalg.norm(a) + np.linalg.norm(b) + 1e-12)

errX = rel_err(dL_dX, gX_num)
errW = rel_err(dL_dW, gW_num)
errb = rel_err(dL_db, gb_num)

assert errX < 2e-5, f"dL_dX failed finite-diff check (rel err {errX:.2e})"
assert errW < 2e-5, f"dL_dW failed finite-diff check (rel err {errW:.2e})"
assert errb < 2e-5, f"dL_db failed finite-diff check (rel err {errb:.2e})"

print("+++ Task 2.4b checks passed.")

+++ Task 2.4b checks passed.


## Task 3.1a — Explain the Reverse Topological Order

Reverse topological order is required because each node in a computational graph depends on values computed earlier, while its gradient depends on contributions from nodes later in the graph. In the backward pass, a node can only compute its gradient after receiving all gradient contributions from the nodes that use its output. Since the graph is a directed acyclic graph, these dependencies flow from inputs to outputs in the forward pass, so gradient information must flow from outputs back to inputs in the reverse pass. This ensures that every local gradient is computed only after all downstream effects have been accumulated.

## Task 3.1b — Implementation

In this section, we combine the local pullbacks from Section 2 into a complete forward and backward pass.

The forward pass is:

$$
Z = XW + b
$$

$$
H = \text{ReLU}(Z)
$$

$$
\hat{y} = Hv + c
$$

$$
L = \frac{1}{N}\sum_{i=1}^N (\hat{y}_i - y_i)^2
$$

The backward pass is then executed in reverse topological order:

1. Loss pullback
2. Output affine pullback
3. Activation pullback
4. Input affine pullback

This produces the gradients with respect to the parameters $W, b, v,$ and $c$.

In [105]:
import numpy as np

def full_forward_backward(X, y, W, b, v, c):
    # Forward pass
    Z = X @ W + b
    H = np.maximum(Z, 0.0)
    y_hat = H @ v + c
    loss = np.mean((y_hat - y) ** 2)

    # Backward pass
    dL_dyhat = compute_dL_dyhat(y_hat, y)
    dL_dH, dL_dv, dL_dc = backward_output_affine(dL_dyhat, H, v)
    dL_dZ = backward_activation_relu(dL_dH, Z)
    dL_dW, dL_db, dL_dX = backward_input_affine(dL_dZ, X, W)

    return loss, dL_dW, dL_db, dL_dv, dL_dc

In [106]:
def loss_only(X, y, W, b, v, c):
    Z = X @ W + b
    H = np.maximum(Z, 0.0)
    y_hat = H @ v + c
    return np.mean((y_hat - y) ** 2)

def rel_err(a, b):
    return np.linalg.norm(a - b) / (np.linalg.norm(a) + np.linalg.norm(b) + 1e-12)

np.random.seed(10)
N, d, h = 4, 5, 3
X = np.random.randn(N, d)
y = np.random.randn(N)
W = np.random.randn(d, h)
b = np.random.randn(h)
v = np.random.randn(h)
c = np.random.randn()

# student implementation
loss, dW, db, dv, dc = full_forward_backward(X, y, W, b, v, c)

# --- finite difference check ---
eps = 1e-6

def finite_diff_param(param, param_name):
    grad = np.zeros_like(param)
    it = np.nditer(param, flags=["multi_index"], op_flags=["readwrite"])
    while not it.finished:
        idx = it.multi_index
        params_p = dict(W=W.copy(), b=b.copy(), v=v.copy(), c=c)
        params_m = dict(W=W.copy(), b=b.copy(), v=v.copy(), c=c)
        params_p[param_name][idx] += eps
        params_m[param_name][idx] -= eps
        grad[idx] = (
            loss_only(X, y, **params_p) - loss_only(X, y, **params_m)
        ) / (2 * eps)
        it.iternext()
    return grad

# check W
dW_num = finite_diff_param(W, "W")
assert rel_err(dW, dW_num) < 2e-5, "dW failed finite-difference check"

# check b
db_num = finite_diff_param(b, "b")
assert rel_err(db, db_num) < 2e-5, "db failed finite-difference check"

# check v
dv_num = finite_diff_param(v, "v")
assert rel_err(dv, dv_num) < 2e-5, "dv failed finite-difference check"

# check c
c_p = c + eps
c_m = c - eps
dc_num = (
    loss_only(X, y, W, b, v, c_p) - loss_only(X, y, W, b, v, c_m)
) / (2 * eps)

assert abs(dc - dc_num) / (abs(dc) + abs(dc_num) + 1e-12) < 2e-5, "dc failed finite-difference check"

print("+++ Task 3.1b Full reverse composition checks passed.")

+++ Task 3.1b Full reverse composition checks passed.


## Task 4.1a — Conceptual Understanding

Gradient checking is too slow for large neural networks because it requires evaluating the loss multiple times for each parameter. For every parameter, the loss must be computed with a small positive and negative perturbation, leading to two forward passes per parameter. Since modern neural networks can have millions of parameters, this results in extremely high computational cost compared to backpropagation, which computes all gradients efficiently in a single backward pass.

## Task 4.1b — Implementation

We implement a reusable gradient checking function that compares analytical gradients from backpropagation with numerical gradients computed using central differences.

For a chosen parameter tensor, the function:
1. computes the analytical gradient using `full_forward_backward`
2. computes the numerical gradient by perturbing each parameter entry by a small amount
3. compares the two gradients using relative error

A small relative error indicates that the backpropagation implementation is correct.

In [107]:
def gradient_check(X, y, W, b, v, c, param_name, eps=1e-6):
    # 1. Get analytical gradients from backprop
    loss, dW, db, dv, dc = full_forward_backward(X, y, W, b, v, c)

    if param_name == "W":
        analytic_grad = dW
        param = W.copy()
    elif param_name == "b":
        analytic_grad = db
        param = b.copy()
    elif param_name == "v":
        analytic_grad = dv
        param = v.copy()
    elif param_name == "c":
        analytic_grad = np.array(dc)
        param = np.array(c)
    else:
        raise ValueError("param_name must be one of: 'W', 'b', 'v', 'c'")

    # 2. Compute numerical gradient using central differences
    numeric_grad = np.zeros_like(param)

    it = np.nditer(param, flags=["multi_index"], op_flags=["readwrite"])
    while not it.finished:
        idx = it.multi_index

        W_p, b_p, v_p, c_p = W.copy(), b.copy(), v.copy(), c
        W_m, b_m, v_m, c_m = W.copy(), b.copy(), v.copy(), c

        if param_name == "W":
            W_p[idx] += eps
            W_m[idx] -= eps
        elif param_name == "b":
            b_p[idx] += eps
            b_m[idx] -= eps
        elif param_name == "v":
            v_p[idx] += eps
            v_m[idx] -= eps
        elif param_name == "c":
            c_p = c + eps
            c_m = c - eps

        loss_p, _, _, _, _ = full_forward_backward(X, y, W_p, b_p, v_p, c_p)
        loss_m, _, _, _, _ = full_forward_backward(X, y, W_m, b_m, v_m, c_m)

        numeric_grad[idx] = (loss_p - loss_m) / (2 * eps)
        it.iternext()

    # 3. Compute relative error
    rel_error = np.linalg.norm(analytic_grad - numeric_grad) / (
        np.linalg.norm(analytic_grad) + np.linalg.norm(numeric_grad) + 1e-12
    )

    return rel_error

In [108]:
import numpy as np

np.random.seed(42)
N, d, h = 3, 4, 2
X = np.random.randn(N, d)
y = np.random.randn(N)
W = np.random.randn(d, h)
b = np.random.randn(h)
v = np.random.randn(h)
c = np.random.randn()

rel_error = gradient_check(X, y, W, b, v, c, param_name="W")

assert rel_error < 2e-5, f"Gradient check failed (rel error {rel_error:.2e})"
print("+++ Task 4.1b Gradient checking passed.")

+++ Task 4.1b Gradient checking passed.


## Task 5.1 — Reverse-Mode Efficiency

Reverse-mode is efficient because the loss function is a single scalar value. This means there is only one final output, so gradients can be propagated backward once through the network. During this process, vector–Jacobian products are used to pass a single upstream gradient through each layer, allowing all parameter gradients to be computed in one backward pass.

In contrast, forward-mode uses Jacobian–vector products and propagates gradients from inputs to outputs. To obtain gradients for all parameters, it would require a separate pass for each parameter direction, making it computationally expensive.

Since neural networks typically have many parameters but only one scalar loss, reverse-mode scales efficiently and is therefore practical for deep learning.

## Task 5.2 — Why Order Reverses

For a composition of functions $f = f_L \circ \cdots \circ f_1$, the Jacobian is

$$
Df = Df_L \cdots Df_1.
$$

In reverse-mode differentiation, we use the transpose of this Jacobian. Using the identity

$$
(AB)^\top = B^\top A^\top,
$$

the order of multiplication reverses:

$$
(Df)^\top = Df_1^\top \cdots Df_L^\top.
$$

This shows that gradients must be propagated in the opposite order of the forward computation, since each linear map is applied after taking the transpose.

When the output is scalar, reverse-mode is efficient because it propagates a single vector backward and avoids constructing the full Jacobian, reducing both memory and computational cost.

## Task 5.3 — Conceptual Challenge

The most challenging part of this assignment was understanding why transposes appear during backpropagation. At first, it was difficult to see the logic behind using transposed matrices instead of the original ones. However, I later understood that transposes arise when linear maps are moved across inner products during the derivation, which is essential for reverse-mode differentiation.

Another challenge was tracking multiple tensor shapes throughout the computations. With many matrices and vectors involved, it was sometimes overwhelming to ensure that all dimensions aligned correctly. This required careful attention, as even small mistakes in shapes could lead to incorrect results.